In [17]:
import os
os.chdir("/content")
if not os.path.exists("llm-personality-conditioning"):
    !git clone -b new-prompt-gen https://github.com/beedle2017/llm-personality-conditioning.git
%cd /content/llm-personality-conditioning
!git pull
!pip install -q bitsandbytes

/content/llm-personality-conditioning
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 4.47 KiB | 2.24 MiB/s, done.
From https://github.com/beedle2017/llm-personality-conditioning
   4cfb6fc..7799c44  new-prompt-gen -> origin/new-prompt-gen
Updating 4cfb6fc..7799c44
Fast-forward
 notebooks/direct_prompting.ipynb | 135 ++++++++++++++++++++++++++++++
 scripts/direct_prompting.py      | 176 +++++++++++++++++++++++++++++++++++++++
 2 files changed, 311 insertions(+)
 create mode 100644 notebooks/direct_prompting.ipynb
 create mode 100644 scripts/direct_prompting.py


In [1]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/direct_prompting

Mounted at /content/drive


In [11]:
import os
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [19]:
import sys, importlib
repo = "/content/llm-personality-conditioning"
sys.path[:0] = [repo, f"{repo}/src", f"{repo}/scripts"]
for p in list(sys.path_importer_cache):
    if p.startswith(repo):
        del sys.path_importer_cache[p]
importlib.invalidate_caches()
print(importlib.util.find_spec("rewriting"))

ModuleSpec(name='rewriting', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7e6108cc4f50>, origin='/content/llm-personality-conditioning/src/rewriting/__init__.py', submodule_search_locations=['/content/llm-personality-conditioning/src/rewriting'])


In [20]:
import sys
sys.path[:0] = ["/content/llm-personality-conditioning/src",
                "/content/llm-personality-conditioning/scripts"]

from rewriting.llm import HFGenerator
import direct_prompting

generator = HFGenerator.load(load_4bit=True)
OUT = "/content/drive/MyDrive/direct_prompting"

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

In [21]:
from direct_prompting import run_experiment

run_experiment(generator, f"{OUT}/pilot.jsonl", limit=10, levels=[1, 5])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


seed 20261026: 150 records


150

In [22]:
import pandas as pd
df = pd.read_json(f"{OUT}/pilot.jsonl", lines=True)
print(df.groupby(["condition", "level"])["score"].agg(["mean", "count"]))
print(df.refusal.mean(), df.extracted.isna().mean())
df[df.condition == "persona"][["item_id", "level", "text", "extracted", "score"]].head(10)

                     mean  count
condition level                 
persona   1.0    0.306122     49
          5.0    0.795918     49
0.0 0.0


,item_id,level,text,extracted,score
1,bb-O-001-01,1.0,"I'll order a mild dish, it's just what I'm use...",Yes.,1.0
2,bb-O-001-01,5.0,"I'll order a mild dish, as I'm not a fan of ex...",Yes.,1.0
4,bb-O-001-08,1.0,I'll just agree with their suggestion without ...,Yes,1.0
5,bb-O-001-08,5.0,I'd be honest with my friends and say that I'm...,Yes,1.0
7,bb-O-002-05,1.0,I'm not going to touch my savings bonds. They'...,No,0.0
8,bb-O-002-05,5.0,I'd invest 10% of my annual income in a modera...,Yes,1.0
10,bb-O-003-02,1.0,"I'll stick to having just one or two drinks, a...",No,0.0
11,bb-O-003-02,5.0,"I'd politely decline the offer of a shot, and ...",No,0.0
13,bb-O-003-09,1.0,"I'd probably have a couple of drinks, maybe a ...",No,0.0
14,bb-O-003-09,5.0,"I'd probably have a few drinks, but I wouldn't...",No,0.0


In [25]:
for t in ["O", "C", "E", "A", "N"]:
    run_experiment(generator, f"{OUT}/test_{t}.jsonl", traits=[t],split="test", confirm_frozen=True, batch_size=16)

seed 20261026: 180 records
seed 20261026: 180 records
seed 20261026: 180 records
seed 20261026: 180 records
seed 20261026: 300 records


In [3]:
import glob
import pandas as pd
OUT = "/content/drive/MyDrive/direct_prompting"
df = pd.concat((pd.read_json(f, lines=True) for f in sorted(glob.glob(f"{OUT}/test_*.jsonl"))),ignore_index=True)
print(len(df), "records")
print(df.condition.value_counts())

persona = df[df.condition == "persona"]
print("refusal rate:      ", persona.refusal.mean().round(3))
print("extraction missing:", persona.extracted.isna().mean().round(3))
print("score missing:     ", persona.score.isna().mean().round(3))   # unclear / invalid answers
print(persona.groupby("target_trait")[["refusal", "score"]].agg(lambda s: s.isna().mean() if s.name == "score" else s.mean()))

# Only items whose trait matches the target trait (for BehaviorBench this is every persona row)
on = persona[persona.item_trait == persona.target_trait]

table = on.pivot_table(index="target_trait", columns="level", values="score", aggfunc="mean").round(3)
table["baseline"] = df[df.condition == "draft"].groupby("item_trait").score.mean().round(3)
print(table)

# One number per trait: rank correlation between level and score
print(on.groupby("target_trait").apply(lambda g: g.level.corr(g.score, method="spearman")).round(3))

base = df[df.condition == "draft"].set_index(["item_id", "seed"]).score.rename("base")
p = persona.join(base, on=["item_id", "seed"])
p["shift"] = p.score - p.base
print(p.pivot_table(index="target_trait", columns="level", values="shift", aggfunc="mean").round(3))

1020 records
condition
persona    850
draft      170
Name: count, dtype: int64
refusal rate:       0.0
extraction missing: 0.0
score missing:      0.013
              refusal  score
target_trait                
A                 0.0  0.000
C                 0.0  0.000
E                 0.0  0.000
N                 0.0  0.044
O                 0.0  0.000
level           1.0    2.0    3.0    4.0    5.0  baseline
target_trait                                             
A             0.300  0.667  1.000  1.000  1.000     1.000
C             0.267  0.400  0.667  0.933  0.933     0.700
E             0.267  0.367  0.900  1.000  1.000     0.900
N             0.918  0.938  0.938  0.917  0.935     0.938
O             0.633  0.567  0.533  0.600  0.700     0.500
target_trait
A    0.605
C    0.550
E    0.652
N    0.007
O    0.048
dtype: float64
level           1.0    2.0    3.0    4.0    5.0
target_trait                                   
A            -0.700 -0.333  0.000  0.000  0.000
C          

/tmp/ipykernel_6995/3608887662.py:22: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  print(on.groupby("target_trait").apply(lambda g: g.level.corr(g.score, method="spearman")).round(3))
